# Movie Discovery Assistant — Demo

Chạy interactive chat với toàn bộ pipeline.

**Setup:**
```bash
pip install anthropic pandas numpy scipy scikit-learn
export LLM_API_KEY=your_key_here
```


## 0. Setup


In [1]:
import sys
import os

# notebooks/ nằm một cấp dưới project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from config import DATA_DIR
from src.data_layer import load_data
from src.pipeline import run_pipeline
from src.memory import ConversationMemory
from src.query_classifier import classify_query

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}')


c:\Users\ADMIN\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Sliding Window Attention is enabled but not implemented for `eager`; unexpected results may be encountered.
Some parameters are on the meta device because they were offloaded to the disk and cpu.


Project root : d:\AnhThienLe\Job\rcm_movie
Data dir     : D:\AnhThienLe\Job\rcm_movie\data\ml-latest-small-filtered


## 1. Load Data

Load một lần, dùng cho toàn bộ session.  
TF-IDF matrix mất ~10–20 giây lần đầu.


In [2]:
DS = load_data(DATA_DIR)

density = DS.user_item_matrix.nnz / (
    DS.user_item_matrix.shape[0] * DS.user_item_matrix.shape[1]
)
print(f'Movies         : {len(DS.movies_df):,}')
print(f'Users          : {DS.user_item_matrix.shape[0]}')
print(f'Ratings        : {len(DS.ratings_df):,}')
print(f'TF-IDF shape   : {DS.tfidf_matrix.shape}')
print(f'Matrix density : {density:.4f}  (sparse — expected)')


Loading datasets...
  Movies : 5,135
  Ratings: 74,064 from 610 users
  Tags   : 2,440
  User-item matrix: (610, 5135) (density=0.0236)
  TF-IDF matrix   : (5135, 15000)
Data loaded.

Movies         : 5,135
Users          : 610
Ratings        : 74,064
TF-IDF shape   : (5135, 15000)
Matrix density : 0.0236  (sparse — expected)


## 2. Test Users

| User | Ratings | Avg  | Profile                           |
|------|---------|------|-----------------------------------|
| 1    | 190     | 4.33 | Dense — Action/Comedy fan         |
| 15   | 85      | 3.55 | Medium — Sci-fi oriented          |
| 30   | 18      | 4.61 | **Sparse** — cold-start test case |


In [3]:
def print_user_profile(user_id):
    ratings = DS.ratings_df[DS.ratings_df['userId'] == user_id]
    if ratings.empty:
        print(f'User {user_id}: not found')
        return
    top5 = (
        ratings.sort_values('rating', ascending=False).head(5)
        .merge(DS.movies_df[['movieId', 'title']], on='movieId', how='left')
    )
    print(f'User {user_id}  |  {len(ratings)} ratings  |  avg={ratings["rating"].mean():.2f}')
    for _, r in top5.iterrows():
        print(f'  {r["rating"]}  {r["title"]}')
    print()

for uid in [1, 15, 30]:
    print_user_profile(uid)


User 1  |  190 ratings  |  avg=4.33
  5.0  Willy Wonka & the Chocolate Factory
  5.0  Terminator, The
  5.0  Henry V
  5.0  Full Metal Jacket
  5.0  Blues Brothers, The

User 15  |  85 ratings  |  avg=3.55
  5.0  Frequency
  5.0  Aliens
  5.0  Star Wars: Episode VI - Return of the Jedi
  5.0  Alien
  5.0  Back to the Future

User 30  |  18 ratings  |  avg=4.61
  5.0  Braveheart
  5.0  Star Trek
  5.0  Shawshank Redemption, The
  5.0  21 Jump Street
  5.0  Inception



## 3. Classifier Smoke Test

Verify mỗi query route đúng intent trước khi chạy full pipeline.


In [4]:
SMOKE_TESTS = [
    ('What should I watch tonight?',
     'content',          'Recommendation — default'),
    ('I want a dark psychological thriller with a twist',
     'content',          'Content search'),
    ('What do people with similar taste think about Pulp Fiction?',
     'CF',               'CF multi-hop'),
    ('Why do you think I would like that?',
     'explain_previous', 'Explain previous turn'),
    ("What's my blind spot? What genres am I missing?",
     'analytics',        'Analytics'),
    ('Explain the plot of Inception',
     'lookup',           'Lookup'),
    ('I liked Toy Story but tired of animated movies',
     'hybrid',           'Hybrid: title + exclude genre'),
]

print('Classifier smoke test:')
print('-' * 60)
for query, expected, desc in SMOKE_TESTS:
    result = classify_query(query, DS, has_previous_turn=True)
    ok     = result.intent == expected
    mark   = '✓' if ok else f'✗ got={result.intent}'
    print(f'  {mark:<22}  {desc}')
    if result.constraints:
        print(f'  {"":22}  constraints={result.constraints}')


Classifier smoke test:
------------------------------------------------------------
  ✗ got=CF                Recommendation — default
  ✓                       Content search
  ✓                       CF multi-hop
                          constraints={'title_reference': 296}
  ✓                       Explain previous turn
  ✓                       Analytics
  ✓                       Lookup
                          constraints={'title_reference': 79132}
  ✓                       Hybrid: title + exclude genre
                          constraints={'exclude_genres': ['Animation'], 'title_reference': 1}


## 4. Full Pipeline — 6 Sample Queries (User 1)

Chạy lần lượt — query 2 (`explain_previous`) cần memory từ query 1.


### Query 1 — Recommendation (intent: content)


In [5]:
USER_ID = 1
memory  = ConversationMemory()

run_pipeline(USER_ID, 'What should I watch tonight?', DS, memory)



User 1: "What should I watch tonight?"
[Classifier] intent=CF | constraints={}

[Assistant]
Based on your profile (userId=1) and collaborative filtering analysis, these films are strongly recommended with predicted score=5.0:

1. **Strange Brew**  
   - Rated 5.0 by User 226 (sim=0.113) and User 414 (sim=0.1119)  
   - Comedy genre aligns with your top-rated films like *Willy Wonka*

2. **Local Hero**  
   - Rated 5.0 by User 469  
   - Contains dark comedy elements similar to *Full Metal Jacket*

3. **Being There**  
   - Rated 5.0 by User 597 and User 469  
   - Psychological drama with themes of identity, which complements your high-rated *Henry V*

4. **Gravity**  
   - Rated 5.0 by User 414 (most similar user with 165 films in common)  
   - High-concept sci-fi action, matching your preference for *Terminator*

5. **State and Main**  
   - Rated 5.0 by User 414  
   - Satirical drama with comedic undertones, similar to *The Blues Brothers*

All candidates are supported by ratings

"Based on your profile (userId=1) and collaborative filtering analysis, these films are strongly recommended with predicted score=5.0:\n\n1. **Strange Brew**  \n   - Rated 5.0 by User 226 (sim=0.113) and User 414 (sim=0.1119)  \n   - Comedy genre aligns with your top-rated films like *Willy Wonka*\n\n2. **Local Hero**  \n   - Rated 5.0 by User 469  \n   - Contains dark comedy elements similar to *Full Metal Jacket*\n\n3. **Being There**  \n   - Rated 5.0 by User 597 and User 469  \n   - Psychological drama with themes of identity, which complements your high-rated *Henry V*\n\n4. **Gravity**  \n   - Rated 5.0 by User 414 (most similar user with 165 films in common)  \n   - High-concept sci-fi action, matching your preference for *Terminator*\n\n5. **State and Main**  \n   - Rated 5.0 by User 414  \n   - Satirical drama with comedic undertones, similar to *The Blues Brothers*\n\nAll candidates are supported by ratings from users with ≥0.11 similarity to you. Let me know if you'd like de

### Query 2 — Explain Previous (intent: explain_previous)

Memory đã có turn 1.  
Classifier detect `explain_previous` → không chạy retrieval mới,  
đọc reasoning trail đã lưu từ turn 1.


In [6]:
run_pipeline(USER_ID, 'Why do you think I would like that?', DS, memory)



User 1: "Why do you think I would like that?"
[Classifier] intent=explain_previous | constraints={}

[Assistant]
Based on the collaborative filtering (CF) reasoning trail, I think you might like these movies because they were highly rated (5.0) by users with significant overlap in your viewing history:  
- **Strange Brew** matched users 226 and 414, who share 82–165 movies with you. Both gave it a 5.0, suggesting alignment with your taste in Action/Adventure/Comedy (your top genres).  
- **Local Hero** and **Being There** were rated 5.0 by users 469 and 597/469 respectively. While the dataset doesn’t explicitly link these to your genres, the high ratings from users with substantial shared preferences (e.g., User 414 also rated *Gravity* and *State and Main* highly) indicate potential overlap in your cinematic preferences.  

No specific plot or genre terms were matched in this recommendation trail, but the user similarity scores and 5.0 ratings from aligned users are key factors here.

'Based on the collaborative filtering (CF) reasoning trail, I think you might like these movies because they were highly rated (5.0) by users with significant overlap in your viewing history:  \n- **Strange Brew** matched users 226 and 414, who share 82–165 movies with you. Both gave it a 5.0, suggesting alignment with your taste in Action/Adventure/Comedy (your top genres).  \n- **Local Hero** and **Being There** were rated 5.0 by users 469 and 597/469 respectively. While the dataset doesn’t explicitly link these to your genres, the high ratings from users with substantial shared preferences (e.g., User 414 also rated *Gravity* and *State and Main* highly) indicate potential overlap in your cinematic preferences.  \n\nNo specific plot or genre terms were matched in this recommendation trail, but the user similarity scores and 5.0 ratings from aligned users are key factors here.'

### Query 3 — Content Search (intent: content)


In [7]:
run_pipeline(
    USER_ID,
    'I want a dark psychological thriller with a twist',
    DS, memory,
)



User 1: "I want a dark psychological thriller with a twist"
[Classifier] intent=content | constraints={}

[Assistant]
Based on the dataset and your profile (userId=1), here are the most relevant recommendations for a dark psychological thriller with a twist:

**1. Bringing Out the Dead (similarity=0.0933)**  
- Dataset average: 3.42 (rated by 74 users)  
- Genres: Drama  
- Plot contains matched terms: 'dark'  
- Note: Follows a burnt-out paramedic during a 72-hour crisis, with psychological tension and existential themes.  

**2. Something Wicked This Way Comes (similarity=0.0936)**  
- Dataset average: 2.9 (rated by 34 users) ⚠️ SPARSE_MOVIE  
- Genres: Mystery/Thriller/Fantasy  
- Plot contains matched terms: 'dark', 'twist'  
- Tags: "carnival" (suggests surreal, unsettling atmosphere).  

**3. Murder by Death (similarity=0.1159)**  
- Dataset average: 2.83 (rated by 18 users) ⚠️ SPARSE_MOVIE  
- Genres: Thriller/Mystery  
- Plot contains matched terms: 'twist' (murder mystery wit

'Based on the dataset and your profile (userId=1), here are the most relevant recommendations for a dark psychological thriller with a twist:\n\n**1. Bringing Out the Dead (similarity=0.0933)**  \n- Dataset average: 3.42 (rated by 74 users)  \n- Genres: Drama  \n- Plot contains matched terms: \'dark\'  \n- Note: Follows a burnt-out paramedic during a 72-hour crisis, with psychological tension and existential themes.  \n\n**2. Something Wicked This Way Comes (similarity=0.0936)**  \n- Dataset average: 2.9 (rated by 34 users) ⚠️ SPARSE_MOVIE  \n- Genres: Mystery/Thriller/Fantasy  \n- Plot contains matched terms: \'dark\', \'twist\'  \n- Tags: "carnival" (suggests surreal, unsettling atmosphere).  \n\n**3. Murder by Death (similarity=0.1159)**  \n- Dataset average: 2.83 (rated by 18 users) ⚠️ SPARSE_MOVIE  \n- Genres: Thriller/Mystery  \n- Plot contains matched terms: \'twist\' (murder mystery with a "dinner and a murder"'

### Query 4 — Multi-hop CF (intent: CF)

Pipeline: tìm similar users → filter ai đã rate Pulp Fiction → aggregate ratings.


In [8]:
run_pipeline(
    USER_ID,
    'What do people with similar taste think about Pulp Fiction?',
    DS, memory,
)



User 1: "What do people with similar taste think about Pulp Fiction?"
[Classifier] intent=CF | constraints={'title_reference': 296}

[Assistant]
People with similar taste to you have generally positive opinions about *Pulp Fiction*. Out of 13 similar users who rated this film, the ratings range from 3.0 to 5.0, with a weighted predicted score for you of 4.6/5.0. Key ratings include:

- User 226 (similarity=0.113): rating = 4.5  
- User 57 (similarity=0.1122): rating = 3.0  
- User 414 (similarity=0.1119): rating = 5.0  
- User 198 (similarity=0.1108): rating = 5.0  
- User 590 (similarity=0.1044): rating = 4.5  
- User 45 (similarity=0.1017): rating = 5.0  
- User 301 (similarity=0.1009): rating = 4.5  
- User 72 (similarity=0.0984): rating = 4.5  
- User 469 (similarity=0.0982): rating = 5.0  
- User 552 (similarity=0.0972): rating = 4.0  
- User 201 (similarity=0.0961): rating = 5.0  
- User 115 (similarity=0.0935): rating = 5.0  
- User 160 (similarity=0.0932): rating = 5.0  

Base

'People with similar taste to you have generally positive opinions about *Pulp Fiction*. Out of 13 similar users who rated this film, the ratings range from 3.0 to 5.0, with a weighted predicted score for you of 4.6/5.0. Key ratings include:\n\n- User 226 (similarity=0.113): rating = 4.5  \n- User 57 (similarity=0.1122): rating = 3.0  \n- User 414 (similarity=0.1119): rating = 5.0  \n- User 198 (similarity=0.1108): rating = 5.0  \n- User 590 (similarity=0.1044): rating = 4.5  \n- User 45 (similarity=0.1017): rating = 5.0  \n- User 301 (similarity=0.1009): rating = 4.5  \n- User 72 (similarity=0.0984): rating = 4.5  \n- User 469 (similarity=0.0982): rating = 5.0  \n- User 552 (similarity=0.0972): rating = 4.0  \n- User 201 (similarity=0.0961): rating = 5.0  \n- User 115 (similarity=0.0935): rating = 5.0  \n- User 160 (similarity=0.0932): rating = 5.0  \n\nBased on these ratings, it seems that *Pulp Fiction* is highly likely to align with your taste.'

### Query 5 — Analytics (intent: analytics)


In [9]:
run_pipeline(
    USER_ID,
    "What's my blind spot? What genres am I missing?",
    DS, memory,
)



User 1: "What's my blind spot? What genres am I missing?"
[Classifier] intent=analytics | constraints={}

[Assistant]
Your blind spots are genres that are popular in the dataset but underrepresented in your viewing history. Based on the analytics:

1. **Drama**  
   - Your exposure: 9.7% of ratings (54 films)  
   - Dataset popularity: 19.6% of all ratings  
   - Gap: **-9.9%** (you’ve rated significantly fewer Drama films than the average user)  

2. **Comedy**  
   - Your exposure: 12.2% of ratings (68 films)  
   - Dataset popularity: 17.9% of all ratings  
   - Gap: **-5.7%** (you’ve rated fewer Comedy films than the average user)  

The dataset suggests these genres are widely appreciated but not sufficiently explored in your history. For example, *Drama* films in the dataset have an average rating of 4.52 (you’ve rated them 4.35), and *Comedy* films average 4.25 (you’ve rated them 4.25). However, the larger gap is in quantity, not quality.  

Would you like recommendations to ad

'Your blind spots are genres that are popular in the dataset but underrepresented in your viewing history. Based on the analytics:\n\n1. **Drama**  \n   - Your exposure: 9.7% of ratings (54 films)  \n   - Dataset popularity: 19.6% of all ratings  \n   - Gap: **-9.9%** (you’ve rated significantly fewer Drama films than the average user)  \n\n2. **Comedy**  \n   - Your exposure: 12.2% of ratings (68 films)  \n   - Dataset popularity: 17.9% of all ratings  \n   - Gap: **-5.7%** (you’ve rated fewer Comedy films than the average user)  \n\nThe dataset suggests these genres are widely appreciated but not sufficiently explored in your history. For example, *Drama* films in the dataset have an average rating of 4.52 (you’ve rated them 4.35), and *Comedy* films average 4.25 (you’ve rated them 4.25). However, the larger gap is in quantity, not quality.  \n\nWould you like recommendations to address these blind spots?'

### Query 6 — Lookup (intent: lookup)


In [10]:
run_pipeline(USER_ID, 'Explain the plot of Inception', DS, memory)



User 1: "Explain the plot of Inception"
[Classifier] intent=lookup | constraints={'title_reference': 79132}

[Assistant]
**Inception (2010)** revolves around a team performing *inception*—planting an idea in a target’s subconscious. The plot is detailed in the dataset as follows:  

Former dream architect **Dominick "Dom" Cobb** (Leonardo DiCaprio) and partner **Arthur** infiltrate the subconscious of **Robert Fischer** to dismantle his father’s energy conglomerate. Their team includes **Eames** (a forger), **Yusuf** (a chemist who creates the sedative for multi-layered dreams), and **Ariadne** (an architect designing dreamscapes). The mission unfolds across three dream levels:  
1. **Yusuf’s rainy downtown** (first level): Fischer is kidnapped.  
2. **Arthur’s hotel** (second level): The team recruits Fischer.  
3. **Eames’ snowy mountain fortress** (third level): Fischer confronts projections of his father.  

Each level uses a "kick" (a trigger to wake up). Time in dreams is 20x sl

'**Inception (2010)** revolves around a team performing *inception*—planting an idea in a target’s subconscious. The plot is detailed in the dataset as follows:  \n\nFormer dream architect **Dominick "Dom" Cobb** (Leonardo DiCaprio) and partner **Arthur** infiltrate the subconscious of **Robert Fischer** to dismantle his father’s energy conglomerate. Their team includes **Eames** (a forger), **Yusuf** (a chemist who creates the sedative for multi-layered dreams), and **Ariadne** (an architect designing dreamscapes). The mission unfolds across three dream levels:  \n1. **Yusuf’s rainy downtown** (first level): Fischer is kidnapped.  \n2. **Arthur’s hotel** (second level): The team recruits Fischer.  \n3. **Eames’ snowy mountain fortress** (third level): Fischer confronts projections of his father.  \n\nEach level uses a "kick" (a trigger to wake up). Time in dreams is 20x slower than reality; death in dreams results in falling into **Limbo** (a subconscious realm where time is even more

## 5. Hybrid Query

`"like X but without genre Y"` → intent = **hybrid**  
Content Engine dùng plot của Toy Story làm query gốc  
+ filter out Animation.


In [11]:
run_pipeline(
    USER_ID,
    'I liked Toy Story but I am tired of animated movies — what else?',
    DS,
    ConversationMemory(),
)



User 1: "I liked Toy Story but I am tired of animated movies — what else?"
[Classifier] intent=hybrid | constraints={'exclude_genres': ['Animation'], 'title_reference': 1}

[Assistant]
Here are data-grounded recommendations based on your preference for *Toy Story* but avoiding animation:

**1. Gravity (Predicted Score: 5.0)**  
- **Why**: User 414 (similarity=0.1119, 165 films in common with you) rated this 5.0.  
- **Genres**: Action/Adventure (aligns with your top genres).  
- **Dataset Note**: 74 users rated this film, average = 4.2.  

**2. Local Hero (Predicted Score: 5.0)**  
- **Why**: User 469 (similarity=0.1108, 55 films in common) rated this 5.0.  
- **Genres**: Adventure (matches your high rating for *Terminator* and *Full Metal Jacket*).  

**3. Strange Brew (Predicted Score: 5.0)**  
- **Why**: Users 226 and 414 (both with >0.11 similarity to you) rated this 5.0.  
- **Genres**: Comedy (you’ve rated 68 comedies with an average of 4.33).  

**4. Being There (Predicted Scor

'Here are data-grounded recommendations based on your preference for *Toy Story* but avoiding animation:\n\n**1. Gravity (Predicted Score: 5.0)**  \n- **Why**: User 414 (similarity=0.1119, 165 films in common with you) rated this 5.0.  \n- **Genres**: Action/Adventure (aligns with your top genres).  \n- **Dataset Note**: 74 users rated this film, average = 4.2.  \n\n**2. Local Hero (Predicted Score: 5.0)**  \n- **Why**: User 469 (similarity=0.1108, 55 films in common) rated this 5.0.  \n- **Genres**: Adventure (matches your high rating for *Terminator* and *Full Metal Jacket*).  \n\n**3. Strange Brew (Predicted Score: 5.0)**  \n- **Why**: Users 226 and 414 (both with >0.11 similarity to you) rated this 5.0.  \n- **Genres**: Comedy (you’ve rated 68 comedies with an average of 4.33).  \n\n**4. Being There (Predicted Score: 5.0)**  \n- **Why**: Users 597 and 469 (matched similarity users) rated this 5.0.  \n- **Genres**: Drama (less common in your history but well-received by similar user

## 6. Personalization Test

Cùng query, 3 users khác nhau.  
Kết quả phải khác nhau — User 30 phải trigger `SPARSE_USER` warning.


In [12]:
QUERY = 'What should I watch tonight?'

for uid, label in [
    (1,  'Dense  / Action fan'),
    (15, 'Medium / Sci-fi'),
    (30, 'Sparse / Cold-start'),
]:
    print(f'\n{"─" * 60}')
    print(f'USER {uid}  ({label})')
    print('─' * 60)
    run_pipeline(uid, QUERY, DS, ConversationMemory())



────────────────────────────────────────────────────────────
USER 1  (Dense  / Action fan)
────────────────────────────────────────────────────────────

User 1: "What should I watch tonight?"
[Classifier] intent=CF | constraints={}

[Assistant]
Based on your profile (userId=1) and collaborative filtering analysis, these 5 films have the highest predicted ratings:

1. **Strange Brew** (predicted=5.0)  
   - 2 similar users (226, 414) rated it 5.0  
   - 0.11+ similarity with you (shared 82-165 films)

2. **Local Hero** (predicted=5.0)  
   - 1 similar user (469) rated it 5.0  
   - User 469 shares ~55 films with you

3. **Being There** (predicted=5.0)  
   - 2 similar users (597, 469) rated it 5.0  
   - User 597 shares ~55 films with you

4. **Gravity** (predicted=5.0)  
   - 1 similar user (414) rated it 5.0  
   - User 414 shares 165 films with you

5. **State and Main** (predicted=5.0)  
   - 1 similar user (414) rated it 5.0  
   - User 414 shares 165 films with you  

All recomme

## 7. Edge Cases

System phải fail **trung thực** — không hallucinate.


In [13]:
# Movie không có trong dataset
# The Matrix vắng mặt do thiếu plot data
# → system trả về error, không mô tả từ LLM knowledge
print('=== Movie not in dataset ===')
run_pipeline(1, 'Tell me about The Matrix', DS, ConversationMemory())


=== Movie not in dataset ===

User 1: "Tell me about The Matrix"
[Classifier] intent=content | constraints={}

[Assistant]
"The Matrix" is not in my dataset — I cannot provide data-grounded information about it.  

However, based on your profile and the dataset's top matches for "matrix," you might consider:  
- **Commando** (similarity=0.3661), an Action|Adventure film with a dataset average of 3.5. Plot snippet: *"Retired Delta Force Operative Colonel John Matrix..."*  

Your highest-rated genres (Action, Adventure) align with this film's genre. Would you like more recommendations like this?


'"The Matrix" is not in my dataset — I cannot provide data-grounded information about it.  \n\nHowever, based on your profile and the dataset\'s top matches for "matrix," you might consider:  \n- **Commando** (similarity=0.3661), an Action|Adventure film with a dataset average of 3.5. Plot snippet: *"Retired Delta Force Operative Colonel John Matrix..."*  \n\nYour highest-rated genres (Action, Adventure) align with this film\'s genre. Would you like more recommendations like this?'

In [14]:
# Query quá vague
# 'good', 'recommend' có IDF thấp → TF-IDF scores gần 0
print('=== Vague query ===')
run_pipeline(1, 'Recommend something good', DS, ConversationMemory())


=== Vague query ===

User 1: "Recommend something good"
[Classifier] intent=content | constraints={}

[Assistant]
None


In [15]:
# explain_previous khi memory rỗng
# has_previous() = False → Classifier không route explain_previous
# → fallback sang content
print('=== Explain previous on empty memory ===')
run_pipeline(1, 'Why do you think I would like that?', DS, ConversationMemory())


=== Explain previous on empty memory ===

User 1: "Why do you think I would like that?"
[Classifier] intent=content | constraints={}

[Assistant]
Based on the [REASONING TRAIL FROM PREVIOUS TURN], here is why I suggested *Think Like a Man*:  

1. **Genre Alignment**: You have rated **68 comedies** with an average of **4.33**, including films like *The Blues Brothers* and *Willy Wonka* (both 5.0). *Think Like a Man* is a Comedy, matching your top genre preference.  

2. **Content Match**: The TF-IDF similarity score of **0.2555** indicates the plot keywords ("think", "advice", "couples") align with your query terms.  

3. **Limited Data Note**: The film has sparse ratings (dataset average = 2.5), so predictions about its quality are less reliable, but genre alignment remains a strong signal.  

No similar users (userId=1’s 190 ratings) were directly referenced for this recommendation, as the reasoning focused on content and genre patterns.


'Based on the [REASONING TRAIL FROM PREVIOUS TURN], here is why I suggested *Think Like a Man*:  \n\n1. **Genre Alignment**: You have rated **68 comedies** with an average of **4.33**, including films like *The Blues Brothers* and *Willy Wonka* (both 5.0). *Think Like a Man* is a Comedy, matching your top genre preference.  \n\n2. **Content Match**: The TF-IDF similarity score of **0.2555** indicates the plot keywords ("think", "advice", "couples") align with your query terms.  \n\n3. **Limited Data Note**: The film has sparse ratings (dataset average = 2.5), so predictions about its quality are less reliable, but genre alignment remains a strong signal.  \n\nNo similar users (userId=1’s 190 ratings) were directly referenced for this recommendation, as the reasoning focused on content and genre patterns.'

## 8. Verbose Mode

Xem intermediate results: classifier output, retrieval keys, context preview.


In [16]:
run_pipeline(
    1,
    'What do people with similar taste think about Pulp Fiction?',
    DS,
    ConversationMemory(),
    verbose=True,
)



User 1: "What do people with similar taste think about Pulp Fiction?"
[Classifier] intent=CF | constraints={'title_reference': 296}
[Retrieval] keys=['type', 'target_movie', 'similar_users', 'ratings', 'predicted_score', 'n_rated', 'confidence_flags']
[Reasoning Trail]
[CF TRAIL]
Analyzing what similar users think of 'Pulp Fiction':
Total similar users found: 20
Similar users who rated 'Pulp Fiction': 13

Ratings from similar users:
  - User 226 (similarity=0.113): rating = 4.5
  - User 57 (similarity=0.1122): rating = 3.0
  - User 414 (similarity=0.1119): rating = 5.0
  - User 198 (similarity=0.1108): rating = 5.0
  - User 590 (similarity=0.1044): rating = 4.5
...
[Context preview]
[USER PROFILE]
userId=1 | 190 ratings | avg=4.33
Top genres: Action (74 films), Adventure (68 films), Comedy (68 films)
Highest rated: Willy Wonka & the Chocolate Factory (5.0), Terminator, The (5.0), Henry V (5.0), Full Metal Jacket (5.0), Blues Brothers, The (5.0)

[REASONING TRAIL]
[CF TRAIL]
Analyzing 

"Based on the dataset, 13 users with similar ratings to you (userId=1) have reviewed *Pulp Fiction*. Their ratings range from 3.0 to 5.0, with most favoring it highly:\n\n- **High praise**: 8 users gave 4.5–5.0 (e.g., User 414, 198, 45, 469, 201, 115, 160 with similarity scores >0.0961).\n- **Mixed feedback**: User 57 (similarity=0.1122) gave a 3.0, and User 552 (similarity=0.0972) gave a 4.0.\n\nYour **weighted predicted score** for this film is **4.6/5.0**, derived from these similar users' ratings. The majority of users who share your preferences (Action/Adventure/Comedy tastes) appear to enjoy *Pulp Fiction*."

In [18]:
run_pipeline(
    1,
    'What comedy movie should I watch?',
    DS,
    ConversationMemory(),
    verbose=False,
)


User 1: "What comedy movie should I watch?"
[Classifier] intent=content | constraints={}

[Assistant]
Based on your profile and the dataset's comedy matches, **Animal Crackers (1930)** is recommended. Here's why:

1. **Genre Alignment**: Fits your top genre (Comedy) and includes "Musical" elements, which often overlap with your taste for classic films like *Willy Wonka* and *The Blues Brothers*.
2. **Rating Signal**: The 4.12 average rating from the dataset (though sparse) is above the typical comedy average of 3.7 in your library.
3. **Plot Fit**: Features a "party" and "investigation" plotline, which aligns with your preference for structured, character-driven comedies (e.g., *Full Metal Jacket*).

**Note**: Only 3 users in your dataset have rated this film (⚠️ SPARSE_MOVIE). If you prefer more data-backed picks, consider exploring classic comedies like *Some Like It Hot* (not in current matches but in your historical ratings).


'Based on your profile and the dataset\'s comedy matches, **Animal Crackers (1930)** is recommended. Here\'s why:\n\n1. **Genre Alignment**: Fits your top genre (Comedy) and includes "Musical" elements, which often overlap with your taste for classic films like *Willy Wonka* and *The Blues Brothers*.\n2. **Rating Signal**: The 4.12 average rating from the dataset (though sparse) is above the typical comedy average of 3.7 in your library.\n3. **Plot Fit**: Features a "party" and "investigation" plotline, which aligns with your preference for structured, character-driven comedies (e.g., *Full Metal Jacket*).\n\n**Note**: Only 3 users in your dataset have rated this film (⚠️ SPARSE_MOVIE). If you prefer more data-backed picks, consider exploring classic comedies like *Some Like It Hot* (not in current matches but in your historical ratings).'